In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install duckdb

# Ablation: was the frozen baseline handicapped by a missing `valuesets` DDL?

During the published training and evaluation runs, `schema/schema.sql` did **not** contain a
`CREATE TABLE valuesets` block. The table existed in both DuckDB databases and every gold query
resolved terminology through it, but the model was never shown its columns.

That asymmetry matters for exactly one model state. Fine-tuned models learned the table's shape
implicitly from thousands of training targets -- they demonstrably know it, reaching 100% (familiar)
and 89.1% (unseen) execution match *without* the DDL. The **frozen baseline** had only the table
name to go on, so part of its 10.2%/31.3% may reflect not knowing the lookup table's columns rather
than an inability to reason about the schema.

This notebook measures that directly: the frozen model, evaluated twice on the same held-out rows --

- **without** -- the exact DDL used for the published runs (`schema/schema.sql`)
- **with** -- the same DDL with `schema/valuesets_ddl.sql` prepended

Everything else is fixed: same evaluation sample (same `split_seed`, same size), same metric code,
same database, same in-memory model. Four legs total (2 arms x 2 conditions).

**Why frozen only.** The frozen model is deterministic under greedy decoding and carries no adapter,
so it is seed-independent -- there is nothing to average over three seeds. And the fine-tuned states
have no open question to answer, and no headroom on the familiar arm, already at 100%. An optional
fine-tuned control can be switched on below, but it is off by default.

**Reading the result.** A large positive delta means the published frozen baseline was penalized by
the missing table description, and the frozen-to-SFT gap in the paper should be read as an upper
bound on what fine-tuning contributed. A small delta means the frozen model could not use the schema
regardless, and the published comparison stands as-is.

**Prerequisites.** Same environment as `sft_train.ipynb` / `rl_train.ipynb`: `ROOT` (Drive folder on
Colab, network volume on RunPod) must contain `rl_train.ipynb`, `schema/` (including
`valuesets_ddl.sql`), `data/`, and `Qwen2.5-Coder-14B-Instruct/`. No training checkpoints are needed
unless the fine-tuned control is enabled. Results are written to `rl_outputs/valuesets_ablation/`.

## Bootstrap: locate `ROOT` and the RL notebook

A minimal version of the standard environment cell — just enough to mount Drive and resolve `ROOT`
so `rl_train.ipynb` can be found. The next cell then executes that notebook's own setup cells, which
redo the full path/copy setup properly.

In [ ]:
import os, shutil

# Must be set before torch initializes its CUDA context -- same rationale as the training notebooks.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ON_COLAB = True
except Exception:
    ON_COLAB = False

ON_RUNPOD = 'RUNPOD_POD_ID' in os.environ

if ON_COLAB:
    _default_root = '/content/drive/MyDrive/projects/FHIRSQL'
elif ON_RUNPOD:
    _default_root = '/workspace/FHIRSQL'
else:
    _default_root = 'C:/dev/fhirsql-phase2'
ROOT = os.environ.get('FHIRSQL_ROOT', _default_root)

# rl_train.ipynb normally sits at the top of ROOT (that's how it's synced for training runs);
# fall back to the working directory for a local checkout.
_candidates = [os.path.join(ROOT, 'rl_train.ipynb'), 'rl_train.ipynb']
RL_NOTEBOOK = next((p for p in _candidates if os.path.exists(p)), None)
if RL_NOTEBOOK is None:
    raise FileNotFoundError(
        'Could not find rl_train.ipynb. Looked in:\n  ' + '\n  '.join(_candidates) +
        '\nUpload it next to the data/ and schema/ folders under ROOT, or set FHIRSQL_ROOT.')

VALUESETS_DDL_PATH = os.path.join(ROOT, 'schema', 'valuesets_ddl.sql')
if not os.path.exists(VALUESETS_DDL_PATH):
    raise FileNotFoundError(f'Missing {VALUESETS_DDL_PATH} -- sync schema/valuesets_ddl.sql to ROOT.')

print('on_colab   =', ON_COLAB, '| on_runpod =', ON_RUNPOD)
print('ROOT       =', ROOT)
print('RL_NOTEBOOK=', RL_NOTEBOOK)

## Reuse the published evaluation code

Rather than re-implementing the metrics — which would risk drifting from what produced the published
numbers — this executes `rl_train.ipynb`'s setup cells directly: environment and paths (Drive/RunPod
copies, heldout DB and JSONL), config, the `FHIRSQLLLM` wrapper, data loading and prompt
construction, and the evaluation helpers (`evaluate_generation`, `make_loader`, `_exec_rows`, …).

Cells that would start training are deliberately excluded. This copies the 14B model to local disk
on first run, so expect it to take a while.

In [ ]:
import json

#   3 env/paths & local copies | 4 imports | 5 CFG | 9 FHIRSQLLLM
#   11 data + prompt + reward/eval helpers | 13 evaluate_generation, make_loader | 15 seeding & utils
SETUP_CELLS = [3, 4, 5, 9, 11, 13, 15]

_nb = json.load(open(RL_NOTEBOOK, encoding='utf-8'))
for _ci in SETUP_CELLS:
    _src = ''.join(_nb['cells'][_ci]['source'])
    print(f'--- executing rl_train.ipynb cell {_ci} ---')
    exec(compile(_src, f'rl_train_cell_{_ci}', 'exec'), globals())

for _name in ('CFG', 'FHIRSQLLLM', 'evaluate_generation', 'build_messages', 'schema_ddl',
              'DEVICE', 'LOCAL_HELDOUT_JSONL', 'LOCAL_HELDOUT_DUCKDB',
              'RL_OUTPUTS_ROOT', 'SFT_OUT_BASE', 'OUT_BASE'):
    assert _name in globals(), f'{_name} not defined -- check SETUP_CELLS against rl_train.ipynb'
print('\nsetup complete')

## Ablation configuration

`sample_size` defaults to 3000 -- the same value the published heldout evaluation used. Because the
sampler and `split_seed` are identical, the **without** legs draw exactly the rows the published
frozen evaluation scored, so they should reproduce `results/heldout_eval/frozen_*.json` almost
exactly. That reproduction is checked automatically at the end and validates the whole harness.
Lowering `sample_size` (e.g. to 750) is roughly 3x faster and still gives a clear directional
answer, but forfeits that check.

In [ ]:
import duckdb, random, time, gc, torch

ABL = dict(
    arms=['unseen', 'familiar'],
    sample_size=750,            # AS RUN for results/valuesets_ablation/. NOT the 3000 used by the
                                # published heldout eval, so the `without` legs do not reproduce
                                # results/heldout_eval/frozen_*.json exactly -- they track it within
                                # sampling noise (PAPER.md 5.6). Set 3000 to enable the exact check.
    include_sft_control=True,   # AS RUN: a fine-tuned control was included. Otherwise off by
                                # default -- fine-tuned models already know the valuesets columns
                                # and sit at ceiling on the familiar arm, so it answers nothing the
                                # published results don't already show.
    control_seed=42,
    out_dir=os.path.join(RL_OUTPUTS_ROOT, 'valuesets_ablation'),
)
os.makedirs(ABL['out_dir'], exist_ok=True)

# ---- the two prompt conditions -------------------------------------------------
DDL_WITHOUT = schema_ddl                      # exactly what the published runs used

_vs = open(VALUESETS_DDL_PATH, encoding='utf-8').read()
_vs = _vs[_vs.index('CREATE TABLE'):].strip()  # drop the explanatory header comment
DDL_WITH = _vs + '\n\n' + schema_ddl

assert 'CREATE TABLE valuesets' not in DDL_WITHOUT, (
    'schema.sql already contains valuesets -- the ablation has no contrast to measure. '
    'schema.sql must stay as it was for the published runs.')
assert 'CREATE TABLE valuesets' in DDL_WITH

print(f'DDL without valuesets: {len(DDL_WITHOUT):,} chars')
print(f'DDL with    valuesets: {len(DDL_WITH):,} chars  (+{len(DDL_WITH) - len(DDL_WITHOUT)})')

# ---- evaluation samples: same RNG, seed and size as the published heldout eval --
heldout_con = duckdb.connect(LOCAL_HELDOUT_DUCKDB, read_only=True)

ABL_SAMPLE = {}
for _arm, _path in LOCAL_HELDOUT_JSONL.items():
    _rows = [json.loads(l) for l in open(_path, encoding='utf-8')]
    ABL_SAMPLE[_arm] = random.Random(CFG['split_seed']).sample(
        _rows, min(ABL['sample_size'], len(_rows)))
    print(f"heldout[{_arm}]: {len(_rows)} rows -> ablation sample {len(ABL_SAMPLE[_arm])}")

MATCHES_PUBLISHED = ABL['sample_size'] == CFG['heldout_eval_gen_sample_size']
print(f"\nsample matches published heldout eval ({CFG['heldout_eval_gen_sample_size']}): {MATCHES_PUBLISHED}")
print(f"legs to run: {len(ABL['arms']) * 2 * (2 if ABL['include_sft_control'] else 1)}")
print('out_dir =', ABL['out_dir'])

## Run

The base model is loaded **once** and evaluated under both prompt conditions before being released,
so the two conditions are scored by the identical in-memory model and the comparison cannot be
contaminated by reload nondeterminism.

Each leg writes its JSON as soon as it finishes and completed legs are skipped on re-run, so an
interrupted session resumes without redoing work.

In [ ]:
from peft import PeftModel

def _leg_path(state, arm, condition):
    return os.path.join(ABL['out_dir'], f'{state}_{arm}_{condition}.json')


def run_state(state, adapter_dir=None):
    """Evaluate one model state under BOTH prompt conditions, on every configured arm."""
    todo = [(a, c) for a in ABL['arms'] for c in ('without', 'with')
            if not os.path.exists(_leg_path(state, a, c))]
    if not todo:
        print(f'[skip] {state}: all legs already computed'); return
    if adapter_dir is not None and not os.path.isdir(adapter_dir):
        print(f'[skip] {state}: no checkpoint at {adapter_dir}'); return

    print(f'\n===== {state} -- {len(todo)} leg(s) =====')
    wrapper   = FHIRSQLLLM(CFG)
    tokenizer = wrapper._load_tokenizer()
    model     = wrapper._load_base_model()
    if adapter_dir is not None:
        model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    model.to(DEVICE); model.eval()

    try:
        for arm, condition in todo:
            ddl = DDL_WITH if condition == 'with' else DDL_WITHOUT
            t0 = time.time()
            res = evaluate_generation(model, tokenizer, ABL_SAMPLE[arm], ddl, CFG, heldout_con)
            res.update(state=state, arm=arm, condition=condition,
                       n_sample=len(ABL_SAMPLE[arm]), schema_ddl_chars=len(ddl),
                       elapsed_s=round(time.time() - t0, 1))
            json.dump(res, open(_leg_path(state, arm, condition), 'w'), indent=2)
            print(f"  [{arm}/{condition:<7}] exec={res['gen_exec_match']:.4f} "
                  f"exact={res['gen_exact_match']:.4f} "
                  f"abstP/R={res['gen_abstention_precision']:.2f}/{res['gen_abstention_recall']:.2f} "
                  f"hardcoded={res['gen_hardcoded_rate']:.4f}  ({res['elapsed_s']:.0f}s)")
    finally:
        del model
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.synchronize(); torch.cuda.empty_cache()


run_state('frozen')

if ABL['include_sft_control']:
    _seed = ABL['control_seed']
    run_state(f'sft_seed{_seed}', os.path.join(SFT_OUT_BASE, f'seed_{_seed}', 'best'))

print('\nablation complete')

## Results

In [ ]:
import glob

legs = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(ABL['out_dir'], '*.json')))]
if not legs:
    print('no ablation results yet -- run the cell above')
else:
    METRICS = ['gen_exec_match', 'gen_exact_match', 'gen_structure_match',
               'gen_abstention_precision', 'gen_abstention_recall', 'gen_hardcoded_rate']

    def find(state, arm, cond):
        return next((r for r in legs if r['state'] == state and r['arm'] == arm
                     and r['condition'] == cond), None)

    states = sorted({r['state'] for r in legs}, key=lambda s: (s != 'frozen', s))
    for arm in ABL['arms']:
        print(f"\n================  heldout[{arm}]  ================")
        print(f"{'state':<14}{'metric':<28}{'without':>10}{'with':>10}{'delta':>10}")
        for state in states:
            a, b = find(state, arm, 'without'), find(state, arm, 'with')
            if not (a and b):
                continue
            label = state
            for m in METRICS:
                print(f"{label:<14}{m:<28}{a[m]:>10.4f}{b[m]:>10.4f}{b[m] - a[m]:>+10.4f}")
                label = ''

    # ---- validity check: does the `without` leg reproduce the published frozen eval? ----
    print("\n================  harness validity check  ================")
    if not MATCHES_PUBLISHED:
        print(f"skipped -- sample_size={ABL['sample_size']} differs from the published "
              f"{CFG['heldout_eval_gen_sample_size']}, so the rows aren't the same set.")
    else:
        pub_dir = os.path.join(RL_OUTPUTS_ROOT, 'heldout_eval')
        for arm in ABL['arms']:
            pub_path = os.path.join(pub_dir, f'frozen_{arm}.json')
            leg = find('frozen', arm, 'without')
            if not (leg and os.path.exists(pub_path)):
                print(f'  {arm}: published frozen result not found at {pub_path}'); continue
            pub = json.load(open(pub_path))
            worst = max(abs(leg[m] - pub[m]) for m in METRICS if m in pub)
            verdict = 'MATCH' if worst < 1e-9 else ('close' if worst < 0.01 else 'DIVERGENT')
            print(f"  {arm:<10} max |ablation_without - published| = {worst:.6f}  -> {verdict}")
        print("  (an exact match confirms this notebook reproduces the published frozen evaluation, "
              "so the `with` legs are a like-for-like comparison)")

    print("\nInterpretation: a large positive delta for `frozen` means the published frozen baseline "
          "was penalized by the missing table description rather than by inability to reason, and "
          "the frozen-to-SFT gap in PAPER.md should be read as an upper bound on fine-tuning's "
          "contribution. A small delta means the frozen model could not use this schema regardless "
          "and the published comparison stands as-is.")